In [ ]:
from bayes_changepoint.stats import NormalInverseGamma
import scipy as sp
import plotly
import plotly.graph_objects as go
from IPython.display import display, HTML
import numpy as np

plotly.offline.init_notebook_mode()
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
))

# Normal-inverse-gamma distribution

Assume we have a prior for the mean $\mu$ of normal distribution, conditioned on hyperparameters $\mu_{0}, n_{0}$ and $\sigma^2$ such that

$$p(\mu \vert \sigma^2, \mu_{0}, n_{0} ) \sim \mathcal{N} \left(\mu_{0}, \frac{\sigma^2}{n_{0}} \right)$$

where $n_{0}$ represents the number of hypothetical pseudo-observations connected with the other hyperparameters. For the $\sigma^2$ parameter we use hyperparameters $\nu_{0}, \sigma_{0}^2$ such that

$$p(\sigma^2 \vert \nu_{0}, \sigma_{0}^2) \sim I_{\chi^2} \left( \nu_{0}, \sigma_{0}^2 \right)$$

that is it have the inverse-chi-squared distribution. Equivalently, we can write

$$I_{\chi^2} \left( \nu_{0}, \sigma_{0}^2 \right) = IG \left(\frac{\nu_{0}}{2}, \frac{\nu_{0}\sigma_{0}^2}{2} \right)$$

where $IG$ denotes the inverse-gamma distribution with parameters $\alpha, \beta$. In this form the whole [joint distribution is known as](https://en.wikipedia.org/wiki/Normal-inverse-gamma_distribution) 

$$p(\mu, \sigma^2 \vert \mu_{0}, n_{0}, \alpha, \beta) \sim N-\Gamma^{-1} \left(\mu_{0}, n_{0}, \alpha, \beta \right)$$

the Normal-inverse-gamma distribution $N-\Gamma^{-1}$ which is also implemented in [`scipy.stats.normal_inverse_gamma`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.normal_inverse_gamma.html).
However, having the dual definition using the parameters of $I_{\chi^2}$ and $\Gamma^{-1}$ will prove useful when performing the posterior inference given the new samples.

In [ ]:
mean = 2.3
n_dof = 7.0
n_pseudoobs = 10
variance = 5.6
dist = NormalInverseGamma(mean=mean, variance=variance, n_degrees_of_freedom=n_dof, n_pseudoobservations=n_pseudoobs)

In [ ]:
samples = np.vstack(dist.rvs(size=10000))
sample_mean = np.mean(samples, axis=1)
sample_variance = np.cov(samples)
print(sample_mean)
print(sample_variance)
print(dist.beta, dist.alpha)
theoretical_mean = [mean, dist.beta/(dist.alpha - 1)]
theoretical_variance = [dist.beta/(dist.alpha - 1)/dist.n_pseudoobs, dist.beta**2/(dist.alpha - 1)**2/(dist.alpha - 2)]
print(theoretical_mean)
print(theoretical_variance)

In [ ]:
fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(x=samples[0, :], y=samples[1, :], mode="markers", name="NIG samples"))
fig_obj.add_vline(x=theoretical_mean[0], annotation={"text": "theoretical mean of x"})
fig_obj.add_hline(y=theoretical_mean[1], annotation={"text": "$\\text{theoretical mean of } \\sigma^2$"})
fig_obj.update_xaxes(title="x")
fig_obj.update_yaxes(title="$\\sigma^2$")
fig_obj.update_layout(title="NIG samples")
fig_obj.show(0)

In [ ]:
true_mean = 5.0
true_sigma = 9.0
true_dist = sp.stats.norm(loc=true_mean, scale=true_sigma)
n_sample = 500
true_samples = true_dist.rvs(size=(n_sample,))

mean_mat = np.zeros(shape=(2, n_sample))

for i_sample in range(n_sample):
    dist.update([true_samples[i_sample]])
    mean_mat[:, i_sample] = dist.get_mean()

In [ ]:
fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=mean_mat[0, :], mode="markers+lines"))
fig_obj.add_trace(go.Scatter(y=true_samples, mode="markers"))
fig_obj.add_hline(y=true_mean)
fig_obj.show()

fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=mean_mat[1, :], mode="markers+lines"))
fig_obj.add_hline(y=true_sigma**2)
fig_obj.show()